# CSA08 Assignment 4 – Task 5: The Same Utility Organised Four Ways

**Student:** Darshan A  
**Register Number:** 1973260011  
**Course Code:** CSA0812  
**Marks:** 7 | **Bloom’s Level:** K5 – Evaluate

## Aim
To organise the same attendance-processing utility in four different ways, extend each version with a report of students absent for three consecutive days, and compare the designs.

## Sample data and common rules
Each daily file contains one record per line in this format:

`register_number,date,status`

- `P` means present and `A` means absent.
- A valid record must contain exactly three fields and a status of `P` or `A`.
- Damaged lines are skipped and recorded in an error list.
- The attendance percentage is `present days / total valid days × 100`.
- A student is listed in the additional report only if their status is `A` on all three dates.

The same three daily input files are used for all four organisations so their results can be compared fairly.


## Task 5.1 – Four Organisations (3 Marks)

The utility is organised as:

(a) **One script:** reading, statistics, report writing and program execution are in one Python file.

(b) **One module imported by a main script:** reusable functions are in `attendance_utils.py`; `main.py` imports and runs them.

(c) **Three modules in one folder:** `reader.py`, `stats.py` and `report.py` separate the three function groups; `main.py` coordinates them.

(d) **A package:** the same three modules are placed inside `attendance/`, with `__init__.py` defining the package interface. `main.py` imports from the package.

The `if __name__ == "__main__":` guard is used in executable scripts so their main routine runs only when the file is executed directly, not when imported.


In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

# Create the same three daily attendance files for all four designs.
base = Path("task5_work")
if base.exists():
    shutil.rmtree(base)
base.mkdir()

daily_records = {
    "day1.txt": (
        "192311001,2026-10-01,P\n"
        "192311002,2026-10-01,A\n"
        "192311003,2026-10-01,A\n"
        "192311004,2026-10-01,P\n"
    ),
    "day2.txt": (
        "192311001,2026-10-02,A\n"
        "192311002,2026-10-02,A\n"
        "192311003,2026-10-02,P\n"
        "192311004,2026-10-02,P\n"
    ),
    "day3.txt": (
        "192311001,2026-10-03,A\n"
        "192311002,2026-10-03,A\n"
        "192311003,2026-10-03,A\n"
        "192311004,2026-10-03,P\n"
        "DAMAGED LINE\n"
    )
}

input_folder = base / "input"
input_folder.mkdir()
for filename, content in daily_records.items():
    (input_folder / filename).write_text(content, encoding="utf-8")

print("Sample attendance files created:")
for path in sorted(input_folder.glob("*.txt")):
    print("-", path.name)


Sample attendance files created:
- day1.txt
- day2.txt
- day3.txt


### Shared processing logic

All four versions use the same processing rules. Each organisation reads the daily files once, stores counts by register number, records damaged lines, calculates percentages, and writes:
1. a monthly attendance report; and
2. a second report listing students absent on all three consecutive dates.


### (a) Organisation 1 – One Script

All functions and the main routine are stored in one file. This is straightforward for a small program, but the file becomes longer as more features are added.


In [2]:
one_script_code = '''
from pathlib import Path
import csv

def read_daily_files(folder):
    counts = {}
    errors = []
    files = sorted(Path(folder).glob("*.txt"))

    for path in files:
        with open(path, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                if not line.strip():
                    continue
                parts = line.strip().split(",")
                if len(parts) != 3 or parts[2].strip() not in ("P", "A"):
                    errors.append((path.name, line_number, line.strip()))
                    continue
                register_number, date, status = [part.strip() for part in parts]
                if not register_number or not date:
                    errors.append((path.name, line_number, line.strip()))
                    continue
                if register_number not in counts:
                    counts[register_number] = {"present": 0, "absent": 0, "statuses": []}
                if status == "P":
                    counts[register_number]["present"] += 1
                else:
                    counts[register_number]["absent"] += 1
                counts[register_number]["statuses"].append(status)

    return counts, errors, len(files)


def calculate_percentages(counts):
    results = {}
    for register_number, values in counts.items():
        total = values["present"] + values["absent"]
        percentage = (values["present"] / total * 100) if total else 0
        results[register_number] = {
            "present": values["present"],
            "absent": values["absent"],
            "percentage": percentage,
            "statuses": values["statuses"]
        }
    return results


def write_reports(results, errors, output_folder):
    output = Path(output_folder)
    output.mkdir(exist_ok=True)

    with open(output / "monthly_report.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Register Number", "Present Days", "Absent Days", "Attendance %"])
        for reg, values in sorted(results.items()):
            writer.writerow([reg, values["present"], values["absent"], f"{values['percentage']:.2f}"])

    with open(output / "absent_three_days.txt", "w", encoding="utf-8") as file:
        for reg, values in sorted(results.items()):
            if len(values["statuses"]) == 3 and all(s == "A" for s in values["statuses"]):
                file.write(reg + "\\n")

    with open(output / "damaged_records.log", "w", encoding="utf-8") as file:
        for filename, line_number, content in errors:
            file.write(f"{filename} | line {line_number} | {content}\\n")


def main():
    counts, errors, file_count = read_daily_files("input")
    results = calculate_percentages(counts)
    write_reports(results, errors, "output")
    print("Daily files processed:", file_count)
    print("Students processed:", len(results))
    print("Damaged records:", len(errors))
    print("Students absent for three consecutive days:")
    print((Path("output") / "absent_three_days.txt").read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
'''

org_a = base / "organisation_a"
org_a.mkdir()
(org_a / "input").mkdir()
for p in input_folder.glob("*.txt"):
    shutil.copy(p, org_a / "input" / p.name)
(org_a / "one_script.py").write_text(one_script_code, encoding="utf-8")

run_a = subprocess.run(
    [sys.executable, "one_script.py"],
    cwd=org_a, text=True, capture_output=True
)
print(run_a.stdout)
if run_a.stderr:
    print(run_a.stderr)


Daily files processed: 3
Students processed: 4
Damaged records: 1
Students absent for three consecutive days:
192311002




### (b) Organisation 2 – One Module Imported by a Main Script

The reusable functions are stored in `attendance_utils.py`. The separate `main.py` imports the functions and controls execution. The `__main__` guard prevents the main routine from running just because the module is imported.


In [3]:
module_code = '''
from pathlib import Path
import csv

def read_daily_files(folder):
    counts, errors = {}, []
    files = sorted(Path(folder).glob("*.txt"))
    for path in files:
        with open(path, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, 1):
                if not line.strip():
                    continue
                parts = line.strip().split(",")
                if len(parts) != 3 or parts[2].strip() not in ("P", "A"):
                    errors.append((path.name, line_number, line.strip()))
                    continue
                reg, date, status = [part.strip() for part in parts]
                if not reg or not date:
                    errors.append((path.name, line_number, line.strip()))
                    continue
                counts.setdefault(reg, {"present": 0, "absent": 0, "statuses": []})
                counts[reg]["present" if status == "P" else "absent"] += 1
                counts[reg]["statuses"].append(status)
    return counts, errors, len(files)

def calculate_percentages(counts):
    results = {}
    for reg, values in counts.items():
        total = values["present"] + values["absent"]
        results[reg] = {
            "present": values["present"],
            "absent": values["absent"],
            "percentage": values["present"] / total * 100 if total else 0,
            "statuses": values["statuses"]
        }
    return results

def write_reports(results, errors, output_folder):
    output = Path(output_folder)
    output.mkdir(exist_ok=True)
    with open(output / "monthly_report.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Register Number", "Present Days", "Absent Days", "Attendance %"])
        for reg, v in sorted(results.items()):
            writer.writerow([reg, v["present"], v["absent"], f"{v['percentage']:.2f}"])
    with open(output / "absent_three_days.txt", "w", encoding="utf-8") as file:
        for reg, v in sorted(results.items()):
            if len(v["statuses"]) == 3 and all(s == "A" for s in v["statuses"]):
                file.write(reg + "\\n")
    with open(output / "damaged_records.log", "w", encoding="utf-8") as file:
        for filename, line_number, content in errors:
            file.write(f"{filename} | line {line_number} | {content}\\n")
'''

main_code = '''
from attendance_utils import read_daily_files, calculate_percentages, write_reports

def main():
    counts, errors, file_count = read_daily_files("input")
    results = calculate_percentages(counts)
    write_reports(results, errors, "output")
    print("Daily files processed:", file_count)
    print("Students processed:", len(results))
    print("Damaged records:", len(errors))
    print("Absent for three consecutive days:")
    from pathlib import Path
    print((Path("output") / "absent_three_days.txt").read_text(encoding="utf-8"))

if __name__ == "__main__":
    main()
'''

org_b = base / "organisation_b"
org_b.mkdir()
(org_b / "input").mkdir()
for p in input_folder.glob("*.txt"):
    shutil.copy(p, org_b / "input" / p.name)
(org_b / "attendance_utils.py").write_text(module_code, encoding="utf-8")
(org_b / "main.py").write_text(main_code, encoding="utf-8")

run_b = subprocess.run(
    [sys.executable, "main.py"],
    cwd=org_b, text=True, capture_output=True
)
print(run_b.stdout)
if run_b.stderr:
    print(run_b.stderr)


Daily files processed: 3
Students processed: 4
Damaged records: 1
Absent for three consecutive days:
192311002




### (c) Organisation 3 – Three Modules in One Folder

The functions are separated according to their responsibilities:
- `reader.py` reads and validates daily files.
- `stats.py` calculates attendance percentages and identifies three-day absences.
- `report.py` writes the output files.
- `main.py` connects the modules and runs the program.

This organisation makes each part easier to test and update independently.


In [4]:
reader_code = '''
from pathlib import Path

def read_daily_files(folder):
    counts, errors = {}, []
    files = sorted(Path(folder).glob("*.txt"))
    for path in files:
        with open(path, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, 1):
                if not line.strip():
                    continue
                parts = line.strip().split(",")
                if len(parts) != 3 or parts[2].strip() not in ("P", "A"):
                    errors.append((path.name, line_number, line.strip()))
                    continue
                reg, date, status = [part.strip() for part in parts]
                if not reg or not date:
                    errors.append((path.name, line_number, line.strip()))
                    continue
                counts.setdefault(reg, {"present": 0, "absent": 0, "statuses": []})
                counts[reg]["present" if status == "P" else "absent"] += 1
                counts[reg]["statuses"].append(status)
    return counts, errors, len(files)
'''

stats_code = '''
def calculate_percentages(counts):
    results = {}
    for reg, values in counts.items():
        total = values["present"] + values["absent"]
        results[reg] = {
            "present": values["present"],
            "absent": values["absent"],
            "percentage": values["present"] / total * 100 if total else 0,
            "statuses": values["statuses"]
        }
    return results

def absent_for_three_days(results):
    return [
        reg for reg, values in sorted(results.items())
        if len(values["statuses"]) == 3 and all(status == "A" for status in values["statuses"])
    ]
'''

report_code = '''
from pathlib import Path
import csv
from stats import absent_for_three_days

def write_reports(results, errors, output_folder):
    output = Path(output_folder)
    output.mkdir(exist_ok=True)
    with open(output / "monthly_report.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Register Number", "Present Days", "Absent Days", "Attendance %"])
        for reg, values in sorted(results.items()):
            writer.writerow([reg, values["present"], values["absent"], f"{values['percentage']:.2f}"])
    with open(output / "absent_three_days.txt", "w", encoding="utf-8") as file:
        for reg in absent_for_three_days(results):
            file.write(reg + "\\n")
    with open(output / "damaged_records.log", "w", encoding="utf-8") as file:
        for filename, line_number, content in errors:
            file.write(f"{filename} | line {line_number} | {content}\\n")
'''

main_three_code = '''
from reader import read_daily_files
from stats import calculate_percentages
from report import write_reports
from pathlib import Path

def main():
    counts, errors, file_count = read_daily_files("input")
    results = calculate_percentages(counts)
    write_reports(results, errors, "output")
    print("Daily files processed:", file_count)
    print("Students processed:", len(results))
    print("Damaged records:", len(errors))
    print("Absent for three consecutive days:")
    print((Path("output") / "absent_three_days.txt").read_text(encoding="utf-8"))

if __name__ == "__main__":
    main()
'''

org_c = base / "organisation_c"
org_c.mkdir()
(org_c / "input").mkdir()
for p in input_folder.glob("*.txt"):
    shutil.copy(p, org_c / "input" / p.name)
for filename, code in {
    "reader.py": reader_code,
    "stats.py": stats_code,
    "report.py": report_code,
    "main.py": main_three_code
}.items():
    (org_c / filename).write_text(code, encoding="utf-8")

run_c = subprocess.run(
    [sys.executable, "main.py"],
    cwd=org_c, text=True, capture_output=True
)
print(run_c.stdout)
if run_c.stderr:
    print(run_c.stderr)


Daily files processed: 3
Students processed: 4
Damaged records: 1
Absent for three consecutive days:
192311002




### (d) Organisation 4 – `attendance` Package

The three modules are placed inside an `attendance/` folder. The `__init__.py` file exposes the package functions. The main program imports the package interface and runs the same processing steps.


In [5]:
# Create the attendance package using the same three module responsibilities.

org_d = base / "organisation_d"
package = org_d / "attendance"
org_d.mkdir()
(org_d / "input").mkdir()
for p in input_folder.glob("*.txt"):
    shutil.copy(p, org_d / "input" / p.name)
package.mkdir()

(package / "__init__.py").write_text(
    "from .reader import read_daily_files\n"
    "from .stats import calculate_percentages, absent_for_three_days\n"
    "from .report import write_reports\n",
    encoding="utf-8"
)

# Package reader module
(package / "reader.py").write_text('''
from pathlib import Path

def read_daily_files(folder):
    counts, errors = {}, []
    files = sorted(Path(folder).glob("*.txt"))
    for path in files:
        with open(path, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, 1):
                if not line.strip():
                    continue
                parts = line.strip().split(",")
                if len(parts) != 3 or parts[2].strip() not in ("P", "A"):
                    errors.append((path.name, line_number, line.strip()))
                    continue
                reg, date, status = [part.strip() for part in parts]
                if not reg or not date:
                    errors.append((path.name, line_number, line.strip()))
                    continue
                counts.setdefault(reg, {"present": 0, "absent": 0, "statuses": []})
                counts[reg]["present" if status == "P" else "absent"] += 1
                counts[reg]["statuses"].append(status)
    return counts, errors, len(files)
''', encoding="utf-8")

# Package statistics module
(package / "stats.py").write_text('''
def calculate_percentages(counts):
    results = {}
    for reg, values in counts.items():
        total = values["present"] + values["absent"]
        results[reg] = {
            "present": values["present"],
            "absent": values["absent"],
            "percentage": values["present"] / total * 100 if total else 0,
            "statuses": values["statuses"]
        }
    return results

def absent_for_three_days(results):
    return [
        reg for reg, values in sorted(results.items())
        if len(values["statuses"]) == 3 and all(status == "A" for status in values["statuses"])
    ]
''', encoding="utf-8")

# Package report module
(package / "report.py").write_text('''
from pathlib import Path
import csv
from .stats import absent_for_three_days

def write_reports(results, errors, output_folder):
    output = Path(output_folder)
    output.mkdir(exist_ok=True)
    with open(output / "monthly_report.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Register Number", "Present Days", "Absent Days", "Attendance %"])
        for reg, values in sorted(results.items()):
            writer.writerow([reg, values["present"], values["absent"], f"{values['percentage']:.2f}"])
    with open(output / "absent_three_days.txt", "w", encoding="utf-8") as file:
        for reg in absent_for_three_days(results):
            file.write(reg + "\\n")
    with open(output / "damaged_records.log", "w", encoding="utf-8") as file:
        for filename, line_number, content in errors:
            file.write(f"{filename} | line {line_number} | {content}\\n")
''', encoding="utf-8")

main_package_code = '''
from attendance import read_daily_files, calculate_percentages, write_reports
from pathlib import Path

def main():
    counts, errors, file_count = read_daily_files("input")
    results = calculate_percentages(counts)
    write_reports(results, errors, "output")
    print("Daily files processed:", file_count)
    print("Students processed:", len(results))
    print("Damaged records:", len(errors))
    print("Absent for three consecutive days:")
    print((Path("output") / "absent_three_days.txt").read_text(encoding="utf-8"))

if __name__ == "__main__":
    main()
'''
(org_d / "main.py").write_text(main_package_code, encoding="utf-8")

run_d = subprocess.run(
    [sys.executable, "main.py"],
    cwd=org_d, text=True, capture_output=True
)
print(run_d.stdout)
if run_d.stderr:
    print(run_d.stderr)


Daily files processed: 3
Students processed: 4
Damaged records: 1
Absent for three consecutive days:
192311002




## Task 5.2 – Change the Requirement (2 Marks)

The new requirement is to produce a second report listing students absent on all three consecutive days.

The feature is added to each organisation as follows:

| Organisation | Files changed to add the report | Number of files changed |
|---|---|---:|
| (a) One script | `one_script.py` | 1 |
| (b) One module + main script | `attendance_utils.py` | 1 |
| (c) Three modules in one folder | `stats.py` and `report.py` | 2 |
| (d) Package | `attendance/stats.py` and `attendance/report.py` | 2 |

The reader does not need to change because it already stores each student's daily statuses. The main script does not need to change because it already calls the report-writing function. The statistics module identifies the students who were absent on all three days, and the report module writes their register numbers to `absent_three_days.txt`.

**Test result:** the sample data contains one student absent on all three days: `192311002`.


## Task 5.3 – Compare the Organisations (2 Marks)

| Organisation | Readability | Reuse | Testing | Naming conflicts |
|---|---|---|---|---|
| One script | Easy to follow when very small; becomes long as features grow. | Low; code is tied to one script. | Functions can be tested, but are mixed in one file. | More likely as many functions and variables share one namespace. |
| One module + main | Better separation between reusable functions and execution. | Good; another program can import the module. | Utility functions can be tested without running `main.py`. | Lower than one script, but the module can still become large. |
| Three modules | Responsibilities are clearly separated. | High; reader, statistics and report functions can be reused separately. | Each module can be tested independently. | Lower because functions are divided by purpose. |
| Package | Clear structure for a larger application. | Very high; modules are grouped under one package name. | Individual modules and the package interface can be tested. | Lower; package-qualified imports make names clearer. |

### Difference between the three import styles

- `import attendance.stats` imports the submodule. A function is accessed with its qualified name, such as `attendance.stats.calculate_percentages(...)`.
- `from attendance import stats` imports the `stats` submodule from the package. A function is accessed as `stats.calculate_percentages(...)`.
- `from attendance.stats import calculate_percentages` imports the function directly, so it can be called as `calculate_percentages(...)`.

The first two styles keep the module name visible and can make the source of a function clearer. The third is shorter, but a directly imported function name can conflict with another name in the current file.


## Alternative Concept and Complexity Analysis

If all functions stayed in one script instead of a package, the program could still work: the reading, statistics and report-writing functions would remain in the same file, and the main routine would call them directly. This avoids package setup and imports, but the file becomes harder to navigate as the program grows.

### Import cost and memory

A package has a one-time import cost when its modules are first loaded. Python stores loaded modules in `sys.modules`, so later imports normally reuse the cached module rather than loading it again. Importing a package uses memory for the package and its loaded modules. A single script avoids the separate package imports, but all its function definitions still occupy memory when the script runs.

### Comparison

| Factor | One script | Package |
|---|---|---|
| Finding a function | Search one potentially long file. | Open the module responsible for that function. |
| Changing attendance rules | May require searching through a large script. | Usually change `stats.py`. |
| Changing input format | Find the file-reading section. | Change `reader.py`. |
| Changing report layout | Find the report-writing section. | Change `report.py`. |
| Import overhead | No internal package import is needed. | Small one-time import and module-loading cost; cached in `sys.modules`. |
| Runtime complexity of processing | O(L) for L input lines. | O(L) for L input lines; modular structure does not change the algorithm. |

The package improves maintainability and reuse, especially when the utility grows. For a very small one-off program, one script may be simpler. The choice depends on the expected size, reuse and future changes.


## Automatic Verification Checks

The final cell checks that all four organisations ran successfully, created the monthly report, created the three-day absence report, and identified the expected student.


In [6]:
# Verify all four organisation outputs.

organisation_folders = [
    base / "organisation_a",
    base / "organisation_b",
    base / "organisation_c",
    base / "organisation_d"
]

for index, folder in enumerate(organisation_folders, start=1):
    output = folder / "output"
    monthly = output / "monthly_report.csv"
    absent = output / "absent_three_days.txt"
    errors = output / "damaged_records.log"

    assert monthly.exists(), f"Organisation {index}: monthly report missing"
    assert absent.exists(), f"Organisation {index}: three-day absence report missing"
    assert errors.exists(), f"Organisation {index}: damaged-record log missing"

    absent_students = absent.read_text(encoding="utf-8").splitlines()
    assert absent_students == ["192311002"], f"Organisation {index}: unexpected absent list"

    print(f"Organisation {index}: all output files verified.")

print("\nAll four organisations passed the verification checks.")
print("Expected student absent for three consecutive days: 192311002")


Organisation 1: all output files verified.
Organisation 2: all output files verified.
Organisation 3: all output files verified.
Organisation 4: all output files verified.

All four organisations passed the verification checks.
Expected student absent for three consecutive days: 192311002


## Task 5 – Conclusion

The attendance utility was implemented in four structures: one script, one reusable module with a main script, three separate modules, and an `attendance` package. Each version processed the same input files and generated the same monthly report, damaged-record log and three-consecutive-day absence report. The one-script design is simplest for a small program, while the package provides the clearest separation, reuse and maintainability for a growing system. The verification checks confirm that all four designs identify the same student absent on all three days.
